<a href="https://colab.research.google.com/github/AArna1211/Image_Augmentation/blob/main/Image_Augmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install albumentations opencv-python pillow --quiet

In [ ]:
# ==============================
# 📌 Image Dataset Augmentation
# ==============================


import os
import cv2
from PIL import Image
import albumentations as A

# === 1. Paths ===
input_dir = "/content/images"            # Folder with your original images
output_dir = "/content/augmented_images" # Folder to save augmented images
os.makedirs(output_dir, exist_ok=True)

# === 2. Define augmentation pipeline ===
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussianBlur(p=0.3),
    A.RandomCrop(width=224, height=224, p=0.5),
    A.ColorJitter(p=0.4),
])

# === 3. Augment Images ===
augmentations_per_image = 5  # how many new versions to generate per image

for img_name in os.listdir(input_dir):
    img_path = os.path.join(input_dir, img_name)

    # Only process valid image files
    if not img_name.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert BGR → RGB

    for i in range(augmentations_per_image):
        augmented = transform(image=image)
        aug_img = augmented["image"]

        out_name = f"{os.path.splitext(img_name)[0]}_aug{i}.jpg"
        out_path = os.path.join(output_dir, out_name)

        Image.fromarray(aug_img).save(out_path)

print(f"✅ Augmentation complete! Check '{output_dir}' for results.")


In [ ]:
# ==========================================
# 📌 Heuristic-Based Smart Image Augmentation
# ==========================================

!pip install albumentations opencv-python pillow matplotlib --quiet

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import albumentations as A

# === 1. Paths ===
input_dir = "/content/images"             # Folder with your original images
output_dir = "/content/augmented_images"  # Where to save augmented versions
os.makedirs(output_dir, exist_ok=True)

# === 2. Heuristic functions ===
def is_dark(image, threshold=100):
    """Check if image is dark (mean pixel value below threshold)."""
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    return np.mean(gray) < threshold

def is_blurry(image, threshold=100):
    """Check if image is blurry using variance of Laplacian."""
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    fm = cv2.Laplacian(gray, cv2.CV_64F).var()
    return fm < threshold

def low_color_variance(image, threshold=500):
    """Check if image has low color variance (dull colors)."""
    return np.var(image) < threshold

# === 3. Define augmentation pipelines ===
brighten_pipeline = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=0.4, contrast_limit=0.2, p=1.0),
    A.ColorJitter(p=0.5),
])

sharpen_pipeline = A.Compose([
    A.Sharpen(p=1.0),
    A.GaussianBlur(p=0.3),
])

color_pipeline = A.Compose([
    A.ColorJitter(p=1.0),
    A.HueSaturationValue(p=0.5),
])

default_pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
])

# === 4. Process images ===
augmentations_per_image = 3

for img_name in os.listdir(input_dir):
    if not img_name.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    img_path = os.path.join(input_dir, img_name)
    image = cv2.imread(img_path)
    if image is None:
        continue
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Decide pipeline based on heuristics
    if is_dark(image):
        pipeline = brighten_pipeline
    elif is_blurry(image):
        pipeline = sharpen_pipeline
    elif low_color_variance(image):
        pipeline = color_pipeline
    else:
        pipeline = default_pipeline

    for i in range(augmentations_per_image):
        augmented = pipeline(image=image)
        aug_img = augmented["image"]

        out_name = f"{os.path.splitext(img_name)[0]}_aug{i}.jpg"
        Image.fromarray(aug_img).save(os.path.join(output_dir, out_name))

print(f"✅ Augmentation complete! Saved in {output_dir}")


# === 5. Preview some results ===
def preview_images(original_path, aug_dir, n=3):
    """Preview original + augmented images side by side."""
    original = Image.open(original_path)
    base = os.path.splitext(os.path.basename(original_path))[0]
    aug_imgs = [Image.open(os.path.join(aug_dir, f))
                for f in os.listdir(aug_dir) if f.startswith(base)][:n]

    plt.figure(figsize=(12,4))
    plt.subplot(1, n+1, 1)
    plt.imshow(original)
    plt.axis("off")
    plt.title("Original")

    for i, aug in enumerate(aug_imgs, start=2):
        plt.subplot(1, n+1, i)
        plt.imshow(aug)
        plt.axis("off")
        plt.title(f"Aug {i-1}")
    plt.show()

# Example: preview first image
sample_img = os.path.join(input_dir, os.listdir(input_dir)[0])
preview_images(sample_img, output_dir)


In [ ]:
# ==========================================
# 📌 Smart Augmentation Suggestion (Gradio UI)
# ==========================================

!pip install albumentations opencv-python pillow gradio --quiet

import cv2
import numpy as np
import albumentations as A
from PIL import Image
import gradio as gr

# === 1. Heuristic functions ===
def is_dark(image, threshold=100):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    return np.mean(gray) < threshold

def is_blurry(image, threshold=100):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    fm = cv2.Laplacian(gray, cv2.CV_64F).var()
    return fm < threshold

def low_color_variance(image, threshold=500):
    return np.var(image) < threshold

# === 2. Augmentation pipelines ===
brighten_pipeline = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=0.4, contrast_limit=0.2, p=1.0),
    A.ColorJitter(p=0.5),
])

sharpen_pipeline = A.Compose([
    A.Sharpen(p=1.0),
    A.GaussianBlur(p=0.3),
])

color_pipeline = A.Compose([
    A.ColorJitter(p=1.0),
    A.HueSaturationValue(p=0.5),
])

default_pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
])

# === 3. Decide augmentation policy ===
def suggest_augmentations(img: Image.Image):
    # Convert PIL -> numpy (RGB)
    image = np.array(img.convert("RGB"))

    if is_dark(image):
        suggestion = "Image is dark → Apply Brightness/Contrast adjustments."
        pipeline = brighten_pipeline
    elif is_blurry(image):
        suggestion = "Image is blurry → Apply Sharpening."
        pipeline = sharpen_pipeline
    elif low_color_variance(image):
        suggestion = "Image has low color variance → Apply Color Jitter."
        pipeline = color_pipeline
    else:
        suggestion = "Image looks fine → Apply Default flips/rotations."
        pipeline = default_pipeline

    # Generate preview augmentations
    previews = []
    for i in range(3):  # show 3 variations
        augmented = pipeline(image=image)
        aug_img = augmented["image"]
        previews.append(Image.fromarray(aug_img))

    return suggestion, previews

# === 4. Gradio UI ===
with gr.Blocks() as demo:
    gr.Markdown("## 🖼️ Smart Augmentation Suggester")
    gr.Markdown("Upload an image and get augmentation suggestions + previews.")

    with gr.Row():
        inp = gr.Image(type="pil", label="Upload Image")

    with gr.Row():
        suggestion = gr.Textbox(label="Suggested Augmentation", interactive=False)

    gallery = gr.Gallery(label="Preview Augmentations", columns=3, height="auto")

    inp.change(fn=suggest_augmentations, inputs=inp, outputs=[suggestion, gallery])

demo.launch(debug=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://150003e2033af08780.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install torch torchvision pillow ftfy regex tqdm transformers --quiet

import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import albumentations as A
import numpy as np
import cv2

# === Load CLIP ===
device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# === Categories for classification ===
categories = ["object", "person", "animal", "text"]

def classify_image(img: Image.Image):
    inputs = processor(text=categories, images=img, return_tensors="pt", padding=True).to(device)
    outputs = model(**inputs)
    logits_per_image = outputs.logits_per_image
    probs = logits_per_image.softmax(dim=1).cpu().detach().numpy()[0]
    return categories[np.argmax(probs)], dict(zip(categories, probs))

# === Define augmentation policies ===
augmentation_policies = {
    "object": A.Compose([
        A.RandomRotate90(p=1.0),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.5),
    ]),
    "person": A.Compose([
        A.RandomBrightnessContrast(p=0.7),
        A.Blur(p=0.3),
    ]),
    "animal": A.Compose([
        A.RandomRotate90(p=0.5),
        A.ColorJitter(p=0.5),
    ]),
    "text": A.Compose([
        A.RandomBrightnessContrast(p=0.7),
        A.OpticalDistortion(p=0.3),
    ]),
}

# === Apply classification + augmentations ===
def suggest_and_augment(img: Image.Image):
    # classify
    category, probs = classify_image(img)
    suggestion = f"Detected category: {category} (confidence {probs[category]:.2f})"

    # augment
    pipeline = augmentation_policies[category]
    np_img = np.array(img.convert("RGB"))
    augmented_imgs = [pipeline(image=np_img)["image"] for _ in range(3)]

    return suggestion, [Image.fromarray(aug) for aug in augmented_imgs]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 623.2 kB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


...

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [ ]:
!pip install torch torchvision pillow ftfy regex tqdm transformers albumentations gradio --quiet

import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import albumentations as A
import numpy as np
import gradio as gr

# =====================
# Load CLIP Model
# =====================
device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

categories = ["object", "person", "animal", "text"]

def classify_image(img: Image.Image):
    """Classify image into broad categories using CLIP."""
    inputs = processor(text=categories, images=img, return_tensors="pt", padding=True).to(device)
    outputs = model(**inputs)
    logits_per_image = outputs.logits_per_image
    probs = logits_per_image.softmax(dim=1).cpu().detach().numpy()[0]
    return categories[np.argmax(probs)], dict(zip(categories, probs))

# =====================
# Augmentation Policies
# =====================
import cv2

augmentation_policies = {
    "object": A.Compose([
        A.RandomRotate90(p=1.0),
        A.HorizontalFlip(p=0.5),
        A.ColorJitter(p=0.5),
        A.VerticalFlip(p=0.3),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=45, p=0.7),
        A.RandomBrightnessContrast(p=0.7),
        A.HueSaturationValue(p=0.5),
        A.MotionBlur(p=0.3),
        A.CLAHE(p=0.3),
    ]),

    "person": A.Compose([
        A.RandomBrightnessContrast(p=0.7),
        A.ColorJitter(p=0.5),
        A.Blur(blur_limit=3, p=0.3),
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
        A.Resize(256, 256, p=0.3),   # keep proportions safe
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5), # mild only
        A.RandomShadow(p=0.3),
        A.RandomFog(fog_coef_lower=0.1, fog_coef_upper=0.3, p=0.2),
    ]),

    "animal": A.Compose([
        A.RandomRotate90(p=0.7),
        A.HorizontalFlip(p=0.5),
        A.ColorJitter(p=0.7),
        A.RandomBrightnessContrast(p=0.7),
        A.RandomGamma(p=0.5),
        A.MotionBlur(p=0.4),
        A.GaussNoise(p=0.3),
        A.ElasticTransform(p=0.3),
    ]),

    "text": A.Compose([
        A.RandomBrightnessContrast(p=0.7),
        A.OpticalDistortion(p=0.5),
        A.GridDistortion(p=0.4),
        A.Perspective(scale=(0.05, 0.1), p=0.5),
        A.MotionBlur(p=0.3),
        A.CLAHE(p=0.3),
        A.GaussNoise(p=0.3),
        A.Downscale(scale_min=0.5, scale_max=0.7, p=0.3),  # simulates low-res scans
    ]),
}


# =====================
# Suggest + Augment
# =====================
def suggest_and_augment(image):
    if image is None:
        return "No image uploaded.", None

    img = Image.fromarray(image.astype("uint8"), "RGB")

    # classify
    category, probs = classify_image(img)
    suggestion = f"Detected category: **{category}** (confidence {probs[category]:.2f})"

    # augment
    pipeline = augmentation_policies[category]
    np_img = np.array(img.convert("RGB"))
    augmented_imgs = [pipeline(image=np_img)["image"] for _ in range(3)]

    return suggestion, [Image.fromarray(aug) for aug in augmented_imgs]

# =====================
# Gradio UI
# =====================
demo = gr.Interface(
    fn=suggest_and_augment,
    inputs=gr.Image(type="numpy", label="Upload an Image"),
    outputs=[
        gr.Markdown(label="Suggestion"),
        gr.Gallery(label="Augmented Previews", columns=3, height="auto")
    ],
    title="Smart Image Augmentation Recommender",
    description="Upload an image. The system detects the category (object / person / animal / text) "
                "and suggests suitable augmentation strategies with previews."
)

demo.launch(debug=True)


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipython-input-3858056430.py:49: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
/tmp/ipython-input-3858056430.py:53: UserWarning: Argument(s) 'fog_coef_lower, fog_coef_upper' are not valid for transform RandomFog
  A.RandomFog(fog_coef_lower=0.1, fog_coef_upper=0.3, p=0.2),
/tmp/ipython-input-3858056430.py:75: UserWarning: Argument(s) 'scale_min, scale_max' are not valid for transform Downscale
  A.Downscale(scale_min=0.5, scale_max=0.7, p=0.3),  # simulates low-res scans


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://55de0e29b3bd8187fe.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://55de0e29b3bd8187fe.gradio.live


In [ ]:
!pip install torch torchvision pillow ftfy regex tqdm transformers albumentations gradio --quiet

import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import albumentations as A
import numpy as np
import gradio as gr

# =====================
# Load CLIP
# =====================
device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# =====================
# Augmentation policies & descriptions
# =====================
augmentation_policies = {
    "object": A.Compose([
        A.RandomRotate90(p=1.0),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=45, p=0.7),
        A.RandomBrightnessContrast(p=0.7),
        A.HueSaturationValue(p=0.5),
        A.MotionBlur(p=0.3),
        A.CLAHE(p=0.3),
    ]),
    "person": A.Compose([
        A.RandomBrightnessContrast(p=0.7),
        A.ColorJitter(p=0.5),
        A.Blur(blur_limit=3, p=0.3),
        A.GaussNoise(var_limit=(10.0,50.0), p=0.3),
        A.Resize(256,256,p=0.3),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
        A.RandomShadow(p=0.3),
        A.RandomFog(fog_coef_lower=0.1, fog_coef_upper=0.3, p=0.2),
    ]),
    "animal": A.Compose([
        A.RandomRotate90(p=0.7),
        A.HorizontalFlip(p=0.5),
        A.ColorJitter(p=0.7),
        A.RandomBrightnessContrast(p=0.7),
        A.RandomGamma(p=0.5),
        A.MotionBlur(p=0.4),
        A.GaussNoise(p=0.3),
        A.ElasticTransform(p=0.3),
    ]),
    "text": A.Compose([
        A.RandomBrightnessContrast(p=0.7),
        A.OpticalDistortion(p=0.5),
        A.GridDistortion(p=0.4),
        A.Perspective(scale=(0.05,0.1), p=0.5),
        A.MotionBlur(p=0.3),
        A.CLAHE(p=0.3),
        A.GaussNoise(p=0.3),
        A.Downscale(scale_min=0.5, scale_max=0.7, p=0.3),
    ]),
}

policy_descriptions = {
    "object": "rotate 360 degrees, flip horizontally and vertically, brightness/contrast adjustment",
    "person": "mild rotation, brightness/contrast, blur, noise, mild scaling",
    "animal": "moderate rotation, color jitter, brightness/contrast, noise, elastic transform",
    "text": "perspective distortion, blur, brightness/contrast, noise, downscaling",
}

# =====================
# Suggest top 3 policies
# =====================
def suggest_top_policies(img: Image.Image, top_k=3):
    inputs = processor(
        text=list(policy_descriptions.values()),
        images=img,
        return_tensors="pt",
        padding=True
    ).to(device)
    outputs = model(**inputs)
    logits_per_image = outputs.logits_per_image
    probs = logits_per_image.softmax(dim=1).cpu().detach().numpy()[0]
    top_indices = probs.argsort()[::-1][:top_k]
    top_policies = [(list(policy_descriptions.keys())[i], probs[i]) for i in top_indices]
    return top_policies

# =====================
# Generate previews
# =====================
def generate_previews(img: Image.Image):
    if img is None:
        return "No image uploaded.", None

    np_img = np.array(img.convert("RGB"))
    top_policies = suggest_top_policies(img)

    suggestion_text = "Top 3 Suggested Policies:\n"
    gallery_images = []

    for policy_name, conf in top_policies:
        #suggestion_text += f"- **{policy_name}** (confidence {conf:.2f})\n"
        pipeline = augmentation_policies[policy_name]
        # Generate 1 preview per policy for simplicity
        aug_img = pipeline(image=np_img)["image"]
        gallery_images.append(Image.fromarray(aug_img))

    return suggestion_text, gallery_images

# =====================
# Gradio UI
# =====================
demo = gr.Interface(
    fn=generate_previews,
    inputs=gr.Image(type="pil", label="Upload an Image"),
    outputs=[
        gr.Markdown(label="Suggested Policies"),
        gr.Gallery(label="Augmented Previews", columns=3, height="auto")
    ],
    title="Top-3 Augmentation Policy Recommender",
    description="Upload an image. The system detects top 3 augmentation policies using CLIP and shows example previews."
)

demo.launch(debug=True)


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipython-input-1714955548.py:35: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0,50.0), p=0.3),
/tmp/ipython-input-1714955548.py:39: UserWarning: Argument(s) 'fog_coef_lower, fog_coef_upper' are not valid for transform RandomFog
  A.RandomFog(fog_coef_lower=0.1, fog_coef_upper=0.3, p=0.2),
/tmp/ipython-input-1714955548.py:59: UserWarning: Argument(s) 'scale_min, scale_max' are not valid for transform Downscale
  A.Downscale(scale_min=0.5, scale_max=0.7, p=0.3),


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ffdee419336d9db7d2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://ffdee419336d9db7d2.gradio.live


In [ ]:
!pip install torch torchvision pillow ftfy regex tqdm transformers albumentations gradio --quiet

import torch
from PIL import Image, ImageOps
from transformers import CLIPProcessor, CLIPModel, AutoModelForCausalLM, AutoTokenizer
import albumentations as A
import numpy as np
import gradio as gr

# =====================
# Padding helper
# =====================
def pad_image(img: Image.Image, target_size=(512, 512), color=(0, 0, 0)):
    img = img.convert("RGB")
    w, h = img.size
    new_w, new_h = target_size
    delta_w = max(new_w - w, 0)
    delta_h = max(new_h - h, 0)
    padding = (delta_w // 2, delta_h // 2, delta_w - (delta_w // 2), delta_h - (delta_h // 2))
    return ImageOps.expand(img, padding, fill=color)

# =====================
# Load CLIP
# =====================
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# =====================
# Load LLM (small demo model, replace with your preferred LLM)
# =====================
llm_model_name = "NousResearch/Llama-2-7b-hf"  # Example
tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
llm = AutoModelForCausalLM.from_pretrained(llm_model_name, device_map="auto")

# =====================
# Augmentation policies (your provided version)
# =====================
augmentation_policies = {
    "object": [
        "RandomRotate90", "HorizontalFlip", "VerticalFlip", "ShiftScaleRotate",
        "RandomBrightnessContrast", "HueSaturationValue", "MotionBlur", "CLAHE"
    ],
    "person": [
        "RandomBrightnessContrast", "ColorJitter", "Blur", "GaussNoise",
        "Resize", "ShiftScaleRotate", "RandomShadow", "RandomFog"
    ],
    "animal": [
        "RandomRotate90", "HorizontalFlip", "ColorJitter", "RandomBrightnessContrast",
        "RandomGamma", "MotionBlur", "GaussNoise", "ElasticTransform"
    ],
    "text": [
        "RandomBrightnessContrast", "OpticalDistortion", "GridDistortion", "Perspective",
        "MotionBlur", "CLAHE", "GaussNoise", "Downscale"
    ],
}

# Mapping names to albumentations
def map_name_to_augmentation(name: str):
    mapping = {
        "RandomRotate90": A.RandomRotate90(p=1.0),
        "HorizontalFlip": A.HorizontalFlip(p=1.0),
        "VerticalFlip": A.VerticalFlip(p=1.0),
        "ShiftScaleRotate": A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=45, p=1.0),
        "RandomBrightnessContrast": A.RandomBrightnessContrast(p=1.0),
        "HueSaturationValue": A.HueSaturationValue(p=1.0),
        "MotionBlur": A.MotionBlur(p=1.0),
        "CLAHE": A.CLAHE(p=1.0),
        "ColorJitter": A.ColorJitter(p=1.0),
        "Blur": A.Blur(blur_limit=3, p=1.0),
        "GaussNoise": A.GaussNoise(var_limit=(10.0,50.0), p=1.0),
        "Resize": A.Resize(256, 256, p=1.0),
        "RandomShadow": A.RandomShadow(p=1.0),
        "RandomFog": A.RandomFog(fog_coef_lower=0.1, fog_coef_upper=0.3, p=1.0),
        "RandomGamma": A.RandomGamma(p=1.0),
        "ElasticTransform": A.ElasticTransform(p=1.0),
        "OpticalDistortion": A.OpticalDistortion(p=1.0),
        "GridDistortion": A.GridDistortion(p=1.0),
        "Perspective": A.Perspective(scale=(0.05,0.1), p=1.0),
        "Downscale": A.Downscale(scale_min=0.5, scale_max=0.7, p=1.0),
    }
    return mapping[name]

# =====================
# CLIP classification
# =====================
def classify_image(img: Image.Image):
    labels = ["object", "person", "animal", "text"]
    inputs = clip_processor(text=labels, images=img, return_tensors="pt", padding=True).to(device)
    outputs = clip_model(**inputs)
    logits = outputs.logits_per_image
    probs = logits.softmax(dim=1).cpu().detach().numpy()[0]
    idx = probs.argmax()
    return labels[idx], dict(zip(labels, probs))

# =====================
# LLM augmentation selection
# =====================
def llm_suggest_augmentations(class_name, top_k=3):
    available_ops = augmentation_policies[class_name]
    prompt = f"""
    You are an expert image augmentation policy selector.
    The image belongs to the class: {class_name}.
    Available augmentations are: {", ".join(available_ops)}.
    Suggest the {top_k} most suitable augmentations for this class.
    Only respond with a comma-separated list of augmentation names from the available ones.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)
    outputs = llm.generate(**inputs, max_new_tokens=50)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    suggestions = [s.strip() for s in response.split(",") if s.strip() in available_ops][:top_k]
    return suggestions

# =====================
# Apply selected augmentations
# =====================
def apply_llm_augmentations(img, augmentation_names):
    np_img = np.array(img.convert("RGB"))
    applied_imgs = []
    for aug_name in augmentation_names:
        aug = map_name_to_augmentation(aug_name)
        composed = A.Compose([aug])
        applied_imgs.append(Image.fromarray(composed(image=np_img)["image"]))
    return applied_imgs

# =====================
# Gradio function
# =====================
def process_image(img):
    if img is None:
        return "No image uploaded.", None

    # Step 0: pad
    padded_img = pad_image(img, target_size=(512,512), color=(0,0,0))

    # Step 1: classify
    category, probs = classify_image(padded_img)

    # Step 2: LLM suggests top 3 augmentations
    top_augs = llm_suggest_augmentations(category, top_k=3)

    # Step 3: apply augmentations
    previews = apply_llm_augmentations(padded_img, top_augs)

    # Step 4: output
    suggestion_text = f"Detected category: **{category}** (confidence {probs[category]:.2f})\n"
    suggestion_text += f"LLM suggested augmentations: {', '.join(top_augs)}"
    return suggestion_text, previews

# =====================
# Gradio UI
# =====================
demo = gr.Interface(
    fn=process_image,
    inputs=gr.Image(type="pil", label="Upload an Image"),
    outputs=[
        gr.Markdown(label="Detected Class & Suggested Augmentations"),
        gr.Gallery(label="Augmented Previews", columns=3, height="auto")
    ],
    title="Class-Based LLM Augmentation Recommender",
    description="Upload an image. CLIP detects its class, then LLM recommends top 3 augmentations from that class. Previews are shown."
)

demo.launch(debug=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.2 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b42ebfd36129c3002b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipython-input-3316786780.py:71: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  "GaussNoise": A.GaussNoise(var_limit=(10.0,50.0), p=1.0),
/tmp/ipython-input-3316786780.py:74: UserWarning: Argument(s) 'fog_coef_lower, fog_coef_upper' are not valid for transform RandomFog
  "RandomFog": A.RandomFog(fog_coef_lower=0.1, fog_coef_upper=0.3, p=1.0),
/tmp/ipython-input-3316786780.py:80: UserWarning: Argument(s) 'scale_min, scale_max' are not valid for transform Downscale
  "Downscale": A.Downscale(scale_min=0.5, scale_max=0.7, p=1.0),
/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(sel

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://b42ebfd36129c3002b.gradio.live
